<span style="color: #6a737d; font-family: monospace;">
Created on Fri Apr 04 2025 23:54:46<br>
Author: Mukai (Tom Notch) Yu<br>
Email: mukaiy@andrew.cmu.edu<br>
Affiliation: Carnegie Mellon University, Robotics Institute<br>
<br>
Copyright Ⓒ 2025 Mukai (Tom Notch) Yu<br>
</span>

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# %cd $USF_PROJECT_DIRECTORY doesn't work here because it's set by os.environ, not before the notebook starts
%cd ../..
%load_ext autoreload
%autoreload 2

from copy import deepcopy

import torch
from hydra import compose, initialize
from hydra.utils import instantiate
from pytorch_lightning import Trainer

from usf.network.model.semantic_segmentation import SemanticSegmentationLightningModel
from usf.utils.files import read_file
from usf.utils.image import fig_to_pdf
from usf.utils.torch_numpy import fixed_seed, string_to_seed

In [ ]:
batch_size = 2

## Read Config

In [ ]:
with initialize(
    version_base=None,
    config_path="../../config",
):  # hydra doesn't respect the current working directory
    config = compose(
        config_name="default.yaml",
        overrides=[
            "task=semantic_segmentation",
            "task.data_module.num_workers=1",
            "task.trainer.enable_model_summary=true",
            "task.trainer.log_every_n_steps=1",
            f"task.data_module.batch_size={batch_size}",
            "~task.data_module.train_augmentation",  # disable training augmentation
            "~task.data_module.meta.label.class_weight",  # remove class weight
            "~task.trainer.strategy",  # no strategy
            "~task.trainer.logger",  # disable logging
            "~task.trainer.profiler",  # disable profiler
            "+task.trainer.enable_progress_bar=True",  # print progress in notebook
            "+task.trainer.enable_checkpointing=False",  # disable checkpointing
        ],
    )

## Single Batch Overfit Test

### Setup Datasets

In [ ]:
datamodule = instantiate(config.task.data_module)
datamodule.setup(
    stage="fit"
)  # must setup() because that instantiates self.train_dataset and self.val_dataset

# ensure same sample
with fixed_seed(string_to_seed("Semantic Segmentation")):
    panoramic_single_batch_datamodule = batch_size @ datamodule

panoramic_single_batch_datamodule.val_dataloader = (
    lambda: []
)  # override val_dataloader function to provide empty validation dataloader to skip validation
panoramic_single_batch_datamodule.predict_dataloader = (
    lambda: panoramic_single_batch_datamodule.train_dataloader()
)

In [ ]:
pinhole_single_batch_datamodule = deepcopy(panoramic_single_batch_datamodule)
pinhole_single_batch_datamodule.predict_dataloader = (
    lambda: pinhole_single_batch_datamodule.train_dataloader()
)
pinhole_single_batch_datamodule.train_dataset.set_output_vector(
    read_file("config/lens_normal_map/90_90_280_280.npy")
)

In [ ]:
fisheye_single_batch_datamodule = deepcopy(panoramic_single_batch_datamodule)
fisheye_single_batch_datamodule.predict_dataloader = (
    lambda: fisheye_single_batch_datamodule.train_dataloader()
)
fisheye_single_batch_datamodule.train_dataset.set_output_vector(
    read_file("config/lens_normal_map/180_180_560_560.npy")
).set_output_vector_mask(read_file("config/lens_normal_map/180_180_560_560_mask.npy"))

In [ ]:
train_single_batch_datamodule = fisheye_single_batch_datamodule

### Setup Model

In [ ]:
with initialize(version_base=None, config_path="../../config"):
    planar_config = compose(
        config_name="default.yaml",
        overrides=[
            "task=semantic_segmentation",
            "task/semantic_segmentation@task.architecture=planar",
            "task.architecture.backbone=DeepLabV3",
            "task.architecture.load_pretrain=false",
        ],
    )
    planar_model: SemanticSegmentationLightningModel = instantiate(
        planar_config.task.model
    )

In [ ]:
with initialize(version_base=None, config_path="../../config"):
    spherical_config = compose(
        config_name="default.yaml",
        overrides=[
            "task=semantic_segmentation",
            "task/semantic_segmentation@task.architecture=spherical",
            "task.architecture.backbone=DeepLabV3",
        ],
    )
    spherical_model: SemanticSegmentationLightningModel = instantiate(
        spherical_config.task.model
    )

### Train Planar Model

In [ ]:
trainer_kwargs = {
    # "callbacks": [
    #     callbacks.LearningRateMonitor(logging_interval="epoch"),
    #     # FreezeBatchNorm(),
    # ],
    # "logger": loggers.TensorBoardLogger(
    #     save_dir="tensorboard_logs",
    #     name=f"{planar_model.model.backbone.__class__.__name__} Semantic Segmentation",
    # ),
    "max_epochs": 200,
}
trainer: Trainer = instantiate(config.task.trainer, **trainer_kwargs)

In [ ]:
with torch.autograd.set_detect_anomaly(mode=True, check_nan=True):
    trainer.fit(planar_model, train_single_batch_datamodule)

### Train Spherical Model

In [ ]:
# re-instantiate the trainer to reset the state
trainer_kwargs = {
    # "callbacks": [
    #     callbacks.LearningRateMonitor(logging_interval="epoch"),
    #     # FreezeBatchNorm(),
    # ],
    # "logger": loggers.TensorBoardLogger(
    #     save_dir="tensorboard_logs",
    #     name=f"{spherical_model.model.backbone.__class__.__name__} Semantic Segmentation",
    # ),
    "max_epochs": 200,
}
trainer: Trainer = instantiate(config.task.trainer, **trainer_kwargs)

In [ ]:
with torch.autograd.set_detect_anomaly(mode=True, check_nan=True):
    trainer.fit(spherical_model, train_single_batch_datamodule)

## Visualize Ground Truth and Prediction

In [ ]:
inference_model = spherical_model

In [ ]:
test_type = f"{inference_model.model.backbone.__class__.__name__}-fisheye"

### Panoramic

In [ ]:
panoramic_predictions: list[dict] = trainer.predict(
    inference_model,
    panoramic_single_batch_datamodule,
    return_predictions=True,
)

In [ ]:
panoramic_prediction_batch = panoramic_predictions[0]

In [ ]:
panoramic_gt_figures, panoramic_gt_spherical_vis = (
    panoramic_single_batch_datamodule.train_dataset.visualize_batch(
        batch=panoramic_prediction_batch
    )
)

In [ ]:
for i, fig in enumerate(panoramic_gt_figures):
    fig_to_pdf(fig, f"data/single_batch_overfit_test/ss-of-panoramic-gt-{i}.pdf")

In [ ]:
panoramic_predict_figures, panoramic_predict_spherical_vis = (
    panoramic_single_batch_datamodule.train_dataset.visualize_batch(
        batch={
            "inputs": panoramic_prediction_batch["inputs"],
            "labels": panoramic_prediction_batch["predicts"],
            "meta": panoramic_prediction_batch["meta"],
        },
    )
)

In [ ]:
for i, fig in enumerate(panoramic_predict_figures):
    fig_to_pdf(
        fig, f"data/single_batch_overfit_test/ss-of-{test_type}-panoramic-{i}.pdf"
    )

In [ ]:
panoramic_metrics = inference_model.benchmark(
    predict_maps=panoramic_prediction_batch["predicts"]["spherical_class_maps"],
    gt_maps=panoramic_prediction_batch["labels"]["spherical_class_maps"],
    ignore_index=panoramic_prediction_batch["meta"]["label"]["ignore_index"],
)
print(panoramic_metrics)

### Pinhole

In [ ]:
with fixed_seed(string_to_seed("Semantic Segmentation")):
    pinhole_predictions: list[dict] = trainer.predict(
        inference_model,
        pinhole_single_batch_datamodule,
        return_predictions=True,
    )

In [ ]:
pinhole_prediction_batch = pinhole_predictions[0]

In [ ]:
pinhole_gt_figures, pinhole_gt_spherical_vis = (
    pinhole_single_batch_datamodule.train_dataset.visualize_batch(
        batch=pinhole_prediction_batch
    )
)

In [ ]:
for i, fig in enumerate(pinhole_gt_figures):
    fig_to_pdf(fig, f"data/single_batch_overfit_test/ss-of-pinhole-gt-{i}.pdf")

In [ ]:
pinhole_predict_figures, pinhole_predict_spherical_vis = (
    pinhole_single_batch_datamodule.train_dataset.visualize_batch(
        batch={
            "inputs": pinhole_prediction_batch["inputs"],
            "labels": pinhole_prediction_batch["predicts"],
            "meta": pinhole_prediction_batch["meta"],
        },
    )
)

In [ ]:
for i, fig in enumerate(pinhole_predict_figures):
    fig_to_pdf(fig, f"data/single_batch_overfit_test/ss-of-{test_type}-pinhole-{i}.pdf")

In [ ]:
pinhole_metrics = inference_model.benchmark(
    predict_maps=pinhole_prediction_batch["predicts"]["spherical_class_maps"],
    gt_maps=pinhole_prediction_batch["labels"]["spherical_class_maps"],
    ignore_index=pinhole_prediction_batch["meta"]["label"]["ignore_index"],
)
print(pinhole_metrics)

### Fisheye

In [ ]:
with fixed_seed(string_to_seed("Semantic Segmentation")):
    fisheye_predictions: list[dict] = trainer.predict(
        inference_model,
        fisheye_single_batch_datamodule,
        return_predictions=True,
    )

In [ ]:
fisheye_prediction_batch = fisheye_predictions[0]

In [ ]:
fisheye_gt_figures, fisheye_gt_spherical_vis = (
    fisheye_single_batch_datamodule.train_dataset.visualize_batch(
        batch=fisheye_prediction_batch
    )
)

In [ ]:
for i, fig in enumerate(fisheye_gt_figures):
    fig_to_pdf(fig, f"data/single_batch_overfit_test/ss-of-fisheye-gt-{i}.pdf")

In [ ]:
fisheye_predict_figures, fisheye_predict_spherical_vis = (
    fisheye_single_batch_datamodule.train_dataset.visualize_batch(
        batch={
            "inputs": fisheye_prediction_batch["inputs"],
            "labels": fisheye_prediction_batch["predicts"],
            "meta": fisheye_prediction_batch["meta"],
        },
    )
)

In [ ]:
for i, fig in enumerate(fisheye_predict_figures):
    fig_to_pdf(fig, f"data/single_batch_overfit_test/ss-of-{test_type}-fisheye-{i}.pdf")

In [ ]:
fisheye_metrics = inference_model.benchmark(
    predict_maps=fisheye_prediction_batch["predicts"]["spherical_class_maps"],
    gt_maps=fisheye_prediction_batch["labels"]["spherical_class_maps"],
    ignore_index=fisheye_prediction_batch["meta"]["label"]["ignore_index"],
)
print(fisheye_metrics)